[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/78_dyson_energy_solution.ipynb)

# Solution: Dyson Ring Energy Transport

Reference solution — interval enumeration + prefix sum.

## 解析

**结论：最优路线最多拐一次弯，收集到的能源仓一定是包含起点的一段连续区间 `[L, R]`；枚举区间 + 前缀和求最大能量。**

### 关键观察：只拐一次弯
在一条直线上来回移动，任何「左-右-左-…」多次折返的路线都可以被一条「先到较近端，再一路扫到较远端」的路线**支配**（走过的点集相同，油耗不会更多）。所以最终收集到的仓必是一个包含 `start` 的区间 `[L, R]`（`L <= start <= R`）。

### 油耗公式
要覆盖 `[L, R]`，最省的走法是先走到离起点近的那一端，再折返扫到远端：

```
cost(L, R) = (R - L) + min(start - L, R - start)
```

即「区间总跨度」加上「较近一侧的距离」（这段要走两遍）。只要 `cost <= fuel` 就可行。

### 求解
1. 把仓按坐标排序，做能量前缀和。
2. 枚举区间左右端点（用仓的坐标做候选端点即可），对每个可行 `[L, R]` 用前缀和 O(1) 取区间能量总和，维护最大值。
3. 别忘了「原地不动」的情形（只收起点处的仓）。

端点只需在仓坐标上枚举，`O(M^2)`；也可用**双指针/滑动窗口**做到 `O(M log M)`（排序主导）：右端右移时，左端只会单调右移。

### 边界
- 无仓 → 0；`fuel = 0` → 只能收起点处的仓；重复坐标的仓能量叠加（前缀和天然处理）。

### 复杂度
排序 `O(M log M)`，枚举 `O(M^2)`（或双指针 `O(M)`）。空间 `O(M)`。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import bisect
from typing import List

In [ ]:
# ✅ SOLUTION

class Solution:
    def max_energy(self, start: int, fuel: int,
                   positions: List[int], energies: List[int]) -> int:
        pts = sorted(zip(positions, energies))
        xs = [p for p, _ in pts]
        es = [e for _, e in pts]
        n = len(xs)
        pre = [0] * (n + 1)                       # prefix sum of energy
        for i in range(n):
            pre[i + 1] = pre[i] + es[i]

        best = 0
        for i in range(n):                        # left endpoint = depot i
            for j in range(i, n):                 # right endpoint = depot j
                lo = min(xs[i], start)
                hi = max(xs[j], start)
                cost = (hi - lo) + min(start - lo, hi - start)
                if cost <= fuel:
                    a = bisect.bisect_left(xs, lo)
                    b = bisect.bisect_right(xs, hi)
                    best = max(best, pre[b] - pre[a])

        # standing still: collect depots exactly at start
        a = bisect.bisect_left(xs, start)
        b = bisect.bisect_right(xs, start)
        best = max(best, pre[b] - pre[a])
        return best

In [ ]:
# Demo
sol = Solution()
print(sol.max_energy(5, 4, [8, 4, 6], [4, 7, 2]))   # 9
print(sol.max_energy(0, 9, [-3, 3], [5, 5]))        # 10
print(sol.max_energy(0, 2, [3], [7]))               # 0

In [ ]:
from torch_judge import check
check('dyson_energy')